# Kiểm thử tiền xử lý hợp đồng

Dữ liệu giả lập; chỉ xử lý text đã trích xuất, không gọi AWS/model. Chọn kernel Python có project trong đường dẫn import. Quy ước chi tiết: `docs/architecture.md`. Mặc định token là ước lượng UTF-8, không phải tokenizer của model.

In [ ]:
from pathlib import Path
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "input_processing" / "pipeline.py").is_file())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from input_processing.pipeline import process_job


In [ ]:
text = "Điều 1. Thanh toán\n1. Bên A thanh toán trong 30 ngày.\nTrang 1/2\fĐiều 2. Thời hạn\nMột năm kể từ ngày ký.\n| Mục | Giá |\n| --- | --- |\n| A | 1000 |"
job = {"job_id": "notebook-demo", "contract_id": "DEMO-001", "version": "1", "text": text,
       "source_metadata": {"document_id": "synthetic"},
       "options": {"max_tokens": 128, "overlap_tokens": 16, "min_tokens": 32}}
result = process_job(job)
assert result["status"] == "ok", result
assert result["stats"]["articles_detected"] == 2
assert all(c["token_count"] <= 128 for c in result["chunks"])
result["stats"], result["warnings"]

## Xem chunk và nguồn

`raw_start/end` trỏ tới toàn dòng trong text gốc; `cleaned_start/end` trỏ vào text đã chuẩn hóa. Đây không phải bounding box PDF. `page` chỉ có khi input có dấu ngắt trang `\f`.

In [ ]:
for chunk in result["chunks"]:
    print(chunk["chunk_id"], chunk["pages"], chunk["token_count"], chunk["heading_path"])
    print(chunk["text"])
    print(chunk["source_spans"])
    print()
assert {p for c in result["chunks"] for p in c["pages"]} == {1, 2}

In [ ]:
assert result["stats"]["chunk_count"] == len(result["chunks"])
assert [c["chunk_index"] for c in result["chunks"]] == list(range(len(result["chunks"])))
assert all(c["source_spans"] for c in result["chunks"])
result["document"], result["source_hash"]


Không gọi model hoặc chuyển sang ContractInput. Tiền xử lý kết thúc ở JSON/chunk; kết nối sang agent đang ngắt.

In [ ]:
invalid = process_job({**job, "options": {"max_tokens": "invalid"}})
assert invalid["status"] == "error"
long_result = process_job({**job, "text": "x" * 1000, "options": {"max_tokens": 64, "overlap_tokens": 0}})
assert long_result["status"] == "ok"
assert all(c["token_count"] <= 64 for c in long_result["chunks"])
assert sum(c["text"].count("x") for c in long_result["chunks"]) == 1000
print("Validation, long text, provenance and manifest checks passed.")